# 07. Hybrid Recommender

## Objective

The goal of this notebook is to combine multiple recommendation signals into a single hybrid movie recommender.

We will combine:

* Content-based similarity from movie genres and tags
* Collaborative-filtering scores from user-rating behavior
* Also-watched/rated similarity from movie co-occurrence

We will:

1. Load the trained recommendation components.
2. Generate scores from each recommendation approach.
3. Normalize the different score types.
4. Combine the scores using weighted blending.
5. Generate hybrid recommendations for a selected user and movie.
6. Test the hybrid recommender with example cases.
7. Save the components required for the final application.

The hybrid model will provide the main recommendation logic for the final Streamlit application.


In [1]:
# Import the libraries needed to build the hybrid recommender
import pandas as pd
import numpy as np

from pathlib import Path

In [2]:
# Define the paths to the processed data and saved recommendation models
PROJECT_DIR = Path.cwd().parent

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
MODELS_DIR = PROJECT_DIR / "models"

print("Processed data:", PROCESSED_DIR)
print("Models:", MODELS_DIR)

Processed data: c:\Users\moham\Desktop\Netflix Movie recomendation system\data\processed
Models: c:\Users\moham\Desktop\Netflix Movie recomendation system\models


### Load movie content data

In [3]:
# Load the processed movie metadata used by the content-based recommender
movies_content = pd.read_csv(
    PROCESSED_DIR / "movies_content.csv"
)

print("Movie metadata loaded successfully.")
print("Shape:", movies_content.shape)

Movie metadata loaded successfully.
Shape: (87585, 6)


### Load the collaborative-filtering data

Now we'll load the filtered ratings dataset. We need it because the hybrid recommender must know which movies a user has already rated, so we don't recommend those movies again.

In [4]:
# Load the filtered ratings used to identify each user's existing movie history
ratings_cf = pd.read_csv(
    PROCESSED_DIR / "ratings_cf.csv"
)

print("Ratings loaded successfully.")
print("Shape:", ratings_cf.shape)

Ratings loaded successfully.
Shape: (29542223, 4)


### Load the TF-IDF vectorizer and matrix

In [5]:
# Load the saved TF-IDF vectorizer and movie-content matrix
import joblib
from scipy.sparse import load_npz

tfidf = joblib.load(
    MODELS_DIR / "tfidf_vectorizer.joblib"
)

tfidf_matrix = load_npz(
    MODELS_DIR / "tfidf_matrix.npz"
)

print("TF-IDF vectorizer loaded:", type(tfidf).__name__)
print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF vectorizer loaded: TfidfVectorizer
TF-IDF matrix shape: (87585, 248779)


### Load the SVD model and mappings

In [6]:
# Load the trained collaborative-filtering model and its user/movie mappings
svd = joblib.load(
    MODELS_DIR / "collaborative_svd.joblib"
)

user_to_index = joblib.load(
    MODELS_DIR / "user_to_index.joblib"
)

movie_to_index = joblib.load(
    MODELS_DIR / "movie_to_index.joblib"
)

user_ids = np.load(
    MODELS_DIR / "user_ids.npy"
)

movie_ids = np.load(
    MODELS_DIR / "movie_ids.npy"
)

print("SVD model loaded:", type(svd).__name__)
print("Users:", len(user_ids))
print("Movies:", len(movie_ids))
print("Latent factors:", svd.n_components)

SVD model loaded: TruncatedSVD
Users: 128255
Movies: 31701
Latent factors: 50


### Rebuild the collaborative user-movie matrix

In [8]:
# Rebuild the sparse user-movie rating matrix needed to recreate the SVD factors
from scipy.sparse import csr_matrix

user_indices = ratings_cf["userId"].map(
    user_to_index
).to_numpy()

movie_indices = ratings_cf["movieId"].map(
    movie_to_index
).to_numpy()

ratings_values = ratings_cf["rating"].to_numpy(
    dtype=np.float32
)

user_movie_matrix = csr_matrix(
    (
        ratings_values,
        (user_indices, movie_indices)
    ),
    shape=(len(user_ids), len(movie_ids))
)

print("User-movie matrix shape:", user_movie_matrix.shape)
print("Stored ratings:", user_movie_matrix.nnz)

User-movie matrix shape: (128255, 31701)
Stored ratings: 29542223


### Recreate user and movie factors

In [9]:
# Recreate the user and movie latent-factor matrices from the saved SVD model
user_factors = svd.transform(
    user_movie_matrix
)

movie_factors = svd.components_.T

print("User factors shape:", user_factors.shape)
print("Movie factors shape:", movie_factors.shape)

User factors shape: (128255, 50)
Movie factors shape: (31701, 50)


## Score Normalization

The three recommendation methods produce different types of scores:

| Model                   | Score meaning                        | Typical scale |
| ----------------------- | ------------------------------------ | ------------- |
| Content-based           | Similarity between movie content     | 0 to 1        |
| Collaborative filtering | Personalized latent preference score | Variable      |
| Also-watched            | Similarity in user-rating behavior   | 0 to 1        |

Because these scores are on different scales, we should not combine their raw values directly.

We will normalize each score to a common **0–1 range** before combining them.

A simple min-max normalization is:

$$
x_{\text{normalized}}
=
\frac{x-x_{\min}}
{x_{\max}-x_{\min}}
$$

After normalization:

* `0` represents the lowest score in the candidate set.
* `1` represents the highest score in the candidate set.
* Values between 0 and 1 represent relative strength within that candidate set.

This allows the three recommendation signals to be combined more fairly.


### Create the normalization function

In [10]:
# Create a function that converts a set of scores to a 0–1 range
def normalize_scores(scores):

    scores = np.asarray(scores, dtype=np.float32)

    min_score = scores.min()
    max_score = scores.max()

    # Avoid division by zero when all scores are identical
    if max_score == min_score:
        return np.zeros_like(scores)

    return (scores - min_score) / (max_score - min_score)

### Test score normalization

In [11]:
# Test the normalization function with a simple set of example scores
test_scores = np.array([2.0, 3.0, 4.0, 5.0])

normalized_test_scores = normalize_scores(
    test_scores
)

print("Original scores:", test_scores)
print("Normalized scores:", normalized_test_scores)

Original scores: [2. 3. 4. 5.]
Normalized scores: [0.         0.33333334 0.6666667  1.        ]


### Define the hybrid weights

In [12]:
# Define the initial weights used to combine the three recommendation signals
CONTENT_WEIGHT = 0.30
COLLAB_WEIGHT = 0.50
ALSO_WATCHED_WEIGHT = 0.20

print("Content weight:", CONTENT_WEIGHT)
print("Collaborative weight:", COLLAB_WEIGHT)
print("Also-watched weight:", ALSO_WATCHED_WEIGHT)
print("Total weight:", CONTENT_WEIGHT + COLLAB_WEIGHT + ALSO_WATCHED_WEIGHT)

Content weight: 0.3
Collaborative weight: 0.5
Also-watched weight: 0.2
Total weight: 1.0


### Build the hybrid recommender

In [20]:
# Create a function that generates normalized hybrid scores for candidate movies
from sklearn.metrics.pairwise import cosine_similarity

def hybrid_recommend(
    user_id,
    movie_id,
    n=10,
    candidate_size=200
):

    # Check that the user exists
    if user_id not in user_to_index:
        return "User ID not found."

    # Check that the movie exists
    if movie_id not in movie_to_index:
        return "Movie ID not found."

    # Find the selected movie in the content dataset
    content_rows = movies_content.index[
        movies_content["movieId"] == movie_id
    ]

    if len(content_rows) == 0:
        return "Movie content not found."

    content_index = content_rows[0]

    # Calculate content similarity
    content_scores_all = cosine_similarity(
        tfidf_matrix[content_index],
        tfidf_matrix
    )[0]

    # Calculate collaborative-filtering scores
    user_index = user_to_index[user_id]

    collaborative_scores_all = (
        user_factors[user_index] @ movie_factors.T
    )

    # Calculate also-watched similarity
    cooccurrence_counts = get_cooccurrence_counts(movie_id)

    selected_movie_users = movie_user_counts.loc[movie_id]

    also_watched_scores_all = (
        cooccurrence_counts
        / np.sqrt(
            selected_movie_users
            * movie_user_counts.reindex(movie_ids_aw).to_numpy()
        )
    )

    also_watched_scores_all[~np.isfinite(also_watched_scores_all)] = 0

    # Find movies already rated by the user
    user_rated_movies = ratings_cf.loc[
        ratings_cf["userId"] == user_id,
        "movieId"
    ].values

    # Create candidate movie IDs from the collaborative-filtering movies
    candidate_movie_ids = movie_ids.copy()

    # Remove movies already rated by the user
    unrated_mask = ~np.isin(
        candidate_movie_ids,
        user_rated_movies
    )

    candidate_movie_ids = candidate_movie_ids[unrated_mask]

    # Remove the selected movie itself
    candidate_movie_ids = candidate_movie_ids[
        candidate_movie_ids != movie_id
    ]

    # Get matrix indices for the candidates
    candidate_indices = np.array([
        movie_to_index[movie]
        for movie in candidate_movie_ids
    ])

    # Keep only the strongest collaborative candidates
    candidate_scores = collaborative_scores_all[candidate_indices]

    strongest_indices = np.argsort(
        candidate_scores
    )[::-1][:candidate_size]

    candidate_movie_ids = candidate_movie_ids[
        strongest_indices
    ]

    candidate_indices = candidate_indices[
        strongest_indices
    ]

    # Get content scores for the candidates
    content_indices = movies_content["movieId"].map(
        dict(
            zip(
                movies_content["movieId"],
                movies_content.index
            )
        )
    ).loc[
        movies_content["movieId"].isin(candidate_movie_ids)
    ].values

    content_score_map = dict(
        zip(
            movies_content["movieId"].loc[
                movies_content["movieId"].isin(candidate_movie_ids)
            ],
            content_scores_all[content_indices]
        )
    )

    content_scores = np.array([
        content_score_map.get(movie, 0.0)
        for movie in candidate_movie_ids
    ])

    # Get collaborative scores
    collaborative_scores = collaborative_scores_all[
        candidate_indices
    ]

    # Get also-watched scores
    also_watched_index_map = {
        movie_id_aw: index
        for index, movie_id_aw in enumerate(movie_ids_aw)
    }

    also_watched_indices = np.array([
        also_watched_index_map.get(movie, -1)
        for movie in candidate_movie_ids
    ])

    also_watched_scores = np.array([
        also_watched_scores_all[index]
        if index != -1 else 0.0
        for index in also_watched_indices
    ])

    # Normalize all three score types
    content_normalized = normalize_scores(
        content_scores
    )

    collaborative_normalized = normalize_scores(
        collaborative_scores
    )

    also_watched_normalized = normalize_scores(
        also_watched_scores
    )

    # Combine the normalized scores
    hybrid_scores = (
        CONTENT_WEIGHT * content_normalized
        + COLLAB_WEIGHT * collaborative_normalized
        + ALSO_WATCHED_WEIGHT * also_watched_normalized
    )

    # Select the final recommendations
    top_indices = np.argsort(
        hybrid_scores
    )[::-1][:n]

    recommended_movie_ids = candidate_movie_ids[
        top_indices
    ]

    # Create the final recommendation table
    recommendations = movies_content[
        movies_content["movieId"].isin(
            recommended_movie_ids
        )
    ][[
        "movieId",
        "title",
        "genres_clean"
    ]].copy()

    score_map = dict(
        zip(
            recommended_movie_ids,
            hybrid_scores[top_indices]
        )
    )

    recommendations["hybrid_score"] = (
        recommendations["movieId"].map(score_map)
    )

    recommendations = (
        recommendations
        .set_index("movieId")
        .loc[recommended_movie_ids]
        .reset_index()
    )

    return recommendations

In [22]:
# Load the saved also-watched components for the hybrid recommender
movie_user_matrix_aw = load_npz(
    MODELS_DIR / "movie_user_matrix.npz"
)

movie_ids_aw = np.load(
    MODELS_DIR / "movie_ids_aw.npy"
)

movie_to_index_aw = joblib.load(
    MODELS_DIR / "movie_to_index_aw.joblib"
)

movie_user_counts = pd.read_csv(
    MODELS_DIR / "movie_user_counts.csv",
    index_col=0
)["user_count"]

print("Also-watched matrix:", movie_user_matrix_aw.shape)
print("Also-watched movies:", len(movie_ids_aw))

Also-watched matrix: (31701, 128255)
Also-watched movies: 31701


### Recreate the also-watched helper

In [23]:
# Recreate the also-watched co-occurrence helper for the hybrid recommender
def get_cooccurrence_counts(movie_id):

    # Find the matrix index of the selected movie
    movie_index = movie_to_index_aw.get(movie_id)

    # Check whether the movie exists
    if movie_index is None:
        return None

    # Get the users who rated the selected movie
    user_indices = movie_user_matrix_aw[movie_index].indices

    # Find all movies rated by those users
    related_movies_matrix = movie_user_matrix_aw[:, user_indices]

    # Count how many of those users rated each movie
    cooccurrence_counts = np.asarray(
        related_movies_matrix.sum(axis=1)
    ).ravel()

    # Remove the selected movie itself
    cooccurrence_counts[movie_index] = 0

    return cooccurrence_counts

### Test the hybrid recommender 
We'll use:
- User: 1
- Movie: 2571 — The Matrix
- Recommendations: 10

In [24]:
# Test the hybrid recommender for User 1 using The Matrix as the reference movie
hybrid_recommend(
    user_id=1,
    movie_id=2571,
    n=10
)

,movieId,title,genres_clean,hybrid_score
0,924,2001: A Space Odyssey (1968),Adventure|Drama|Sci-Fi,0.792166
1,858,"Godfather, The (1972)",Crime|Drama,0.698755
2,2858,American Beauty (1999),Drama|Romance,0.637763
3,1193,One Flew Over the Cuckoo's Nest (1975),Drama,0.619288
4,1214,Alien (1979),Horror|Sci-Fi,0.581140
5,1704,Good Will Hunting (1997),Drama|Romance,0.580195
6,1198,Raiders of the Lost Ark (Indiana Jones and the...,Action|Adventure,0.576815
7,750,Dr. Strangelove or: How I Learned to Stop Worr...,Comedy|War,0.565532
8,1240,"Terminator, The (1984)",Action|Sci-Fi|Thriller,0.560501
9,318,"Shawshank Redemption, The (1994)",Crime|Drama,0.529131


### Inspect the contribution of each signal

In [26]:
# Create a diagnostic function that shows how each signal contributes to the hybrid score
def inspect_hybrid_signals(
    user_id,
    movie_id,
    n=10,
    candidate_size=200
):

    # Check that the user exists
    if user_id not in user_to_index:
        return "User ID not found."

    # Check that the movie exists
    if movie_id not in movie_to_index:
        return "Movie ID not found."

    # Find the selected movie in the content dataset
    content_rows = movies_content.index[
        movies_content["movieId"] == movie_id
    ]

    if len(content_rows) == 0:
        return "Movie content not found."

    content_index = content_rows[0]

    # Calculate content similarity
    content_scores_all = cosine_similarity(
        tfidf_matrix[content_index],
        tfidf_matrix
    )[0]

    # Calculate collaborative scores
    user_index = user_to_index[user_id]

    collaborative_scores_all = (
        user_factors[user_index] @ movie_factors.T
    )

    # Calculate also-watched similarity
    cooccurrence_counts = get_cooccurrence_counts(movie_id)

    selected_movie_users = movie_user_counts.loc[movie_id]

    also_watched_scores_all = (
        cooccurrence_counts
        / np.sqrt(
            selected_movie_users
            * movie_user_counts.reindex(movie_ids_aw).to_numpy()
        )
    )

    also_watched_scores_all[~np.isfinite(also_watched_scores_all)] = 0

    # Find movies already rated by the user
    user_rated_movies = ratings_cf.loc[
        ratings_cf["userId"] == user_id,
        "movieId"
    ].values

    # Create candidate movies
    candidate_movie_ids = movie_ids.copy()

    candidate_movie_ids = candidate_movie_ids[
        ~np.isin(candidate_movie_ids, user_rated_movies)
    ]

    candidate_movie_ids = candidate_movie_ids[
        candidate_movie_ids != movie_id
    ]

    # Get collaborative indices
    candidate_indices = np.array([
        movie_to_index[movie]
        for movie in candidate_movie_ids
    ])

    # Keep the strongest collaborative candidates
    collaborative_candidate_scores = (
        collaborative_scores_all[candidate_indices]
    )

    strongest_indices = np.argsort(
        collaborative_candidate_scores
    )[::-1][:candidate_size]

    candidate_movie_ids = candidate_movie_ids[
        strongest_indices
    ]

    candidate_indices = candidate_indices[
        strongest_indices
    ]

    # Map movie IDs to content-matrix indices
    content_index_map = dict(
        zip(
            movies_content["movieId"],
            movies_content.index
        )
    )

    content_scores = np.array([
        content_scores_all[
            content_index_map[movie]
        ]
        if movie in content_index_map
        else 0.0
        for movie in candidate_movie_ids
    ])

    # Get collaborative scores
    collaborative_scores = collaborative_scores_all[
        candidate_indices
    ]

    # Map movie IDs to also-watched matrix indices
    also_watched_index_map = {
        movie_id_aw: index
        for index, movie_id_aw in enumerate(movie_ids_aw)
    }

    also_watched_scores = np.array([
        also_watched_scores_all[
            also_watched_index_map[movie]
        ]
        if movie in also_watched_index_map
        else 0.0
        for movie in candidate_movie_ids
    ])

    # Normalize the three signals
    content_normalized = normalize_scores(
        content_scores
    )

    collaborative_normalized = normalize_scores(
        collaborative_scores
    )

    also_watched_normalized = normalize_scores(
        also_watched_scores
    )

    # Calculate the weighted hybrid score
    hybrid_scores = (
        CONTENT_WEIGHT * content_normalized
        + COLLAB_WEIGHT * collaborative_normalized
        + ALSO_WATCHED_WEIGHT * also_watched_normalized
    )

    # Select the final recommendations
    top_indices = np.argsort(
        hybrid_scores
    )[::-1][:n]

    recommended_movie_ids = candidate_movie_ids[
        top_indices
    ]

    # Create the diagnostic table
    recommendations = movies_content[
        movies_content["movieId"].isin(
            recommended_movie_ids
        )
    ][[
        "movieId",
        "title",
        "genres_clean"
    ]].copy()

    recommendations["content_score"] = (
        content_normalized[top_indices]
    )

    recommendations["collaborative_score"] = (
        collaborative_normalized[top_indices]
    )

    recommendations["also_watched_score"] = (
        also_watched_normalized[top_indices]
    )

    recommendations["hybrid_score"] = (
        hybrid_scores[top_indices]
    )

    # Preserve recommendation order
    recommendations = (
        recommendations
        .set_index("movieId")
        .loc[recommended_movie_ids]
        .reset_index()
    )

    return recommendations

In [27]:
inspect_hybrid_signals(
    user_id=1,
    movie_id=2571,
    n=10
)

,movieId,title,genres_clean,content_score,collaborative_score,also_watched_score,hybrid_score
0,924,2001: A Space Odyssey (1968),Adventure|Drama|Sci-Fi,0.120402,0.877847,0.721220,0.619288
1,858,"Godfather, The (1972)",Crime|Drama,0.355880,0.677612,0.960964,0.637763
2,2858,American Beauty (1999),Drama|Romance,0.292135,0.482981,1.000000,0.529131
3,1193,One Flew Over the Cuckoo's Nest (1975),Drama,0.349086,0.615737,0.842730,0.581140
4,1214,Alien (1979),Horror|Sci-Fi,0.093191,0.705063,0.981629,0.576815
5,1704,Good Will Hunting (1997),Drama|Romance,0.646684,0.372223,0.901921,0.560501
6,1198,Raiders of the Lost Ark (Indiana Jones and the...,Action|Adventure,0.103273,0.749157,0.873176,0.580195
7,750,Dr. Strangelove or: How I Learned to Stop Worr...,Comedy|War,0.067064,1.000000,0.893178,0.698755
8,1240,"Terminator, The (1984)",Action|Sci-Fi|Thriller,0.052441,0.847987,0.629032,0.565532
9,318,"Shawshank Redemption, The (1994)",Crime|Drama,0.718458,0.875458,0.694495,0.792166


Our current weights are:

30% content + 50% collaborative + 20% also-watched

So collaborative filtering has the largest influence.
Right now, the candidate pool is selected from the top 200 collaborative-filtering movies first.
Now I am trying to create the candidate pool from the top candidates from all three models, then let the hybrid score decide the final ranking.

### Create a multi-source candidate pool

In [28]:
# Create a candidate pool using the strongest movies from all three recommendation signals

CANDIDATE_TOP_K = 100

print("Candidate pool setting:", CANDIDATE_TOP_K)

Candidate pool setting: 100


### Build the candidate-generation function.

In [29]:
# Build a candidate pool from the top movies produced by each recommendation signal

def generate_hybrid_candidates(user_id, movie_id, top_k=100):
    if user_id not in user_to_index:
        return "User ID not found."

    if movie_id not in movie_to_index:
        return "Movie ID not found."

    # Content scores
    content_rows = movies_content.index[
        movies_content["movieId"] == movie_id
    ]

    if len(content_rows) == 0:
        return "Movie content not found."

    content_index = content_rows[0]

    content_scores_all = cosine_similarity(
        tfidf_matrix[content_index],
        tfidf_matrix
    )[0]

    # Collaborative scores
    user_index = user_to_index[user_id]

    collaborative_scores_all = (
        user_factors[user_index] @ movie_factors.T
    )

    # Also-watched scores
    cooccurrence_counts = get_cooccurrence_counts(movie_id)

    selected_movie_users = movie_user_counts.loc[movie_id]

    also_watched_scores_all = (
        cooccurrence_counts
        / np.sqrt(
            selected_movie_users
            * movie_user_counts.reindex(movie_ids_aw).to_numpy()
        )
    )

    also_watched_scores_all[~np.isfinite(also_watched_scores_all)] = 0

    # Remove movies already rated by the user
    user_rated_movies = ratings_cf.loc[
        ratings_cf["userId"] == user_id,
        "movieId"
    ].values

    rated_set = set(user_rated_movies)

    # Get top content candidates
    content_movie_ids = movies_content["movieId"].to_numpy()

    valid_content_mask = ~np.isin(
        content_movie_ids,
        list(rated_set)
    )

    valid_content_indices = np.where(valid_content_mask)[0]

    content_top_indices = valid_content_indices[
        np.argsort(
            content_scores_all[valid_content_indices]
        )[::-1][:top_k]
    ]

    content_candidates = set(
        content_movie_ids[content_top_indices]
    )

    # Get top collaborative candidates
    valid_collab_movie_ids = movie_ids[
        ~np.isin(movie_ids, list(rated_set))
    ]

    valid_collab_indices = np.array([
        movie_to_index[movie]
        for movie in valid_collab_movie_ids
    ])

    collab_top_indices = valid_collab_indices[
        np.argsort(
            collaborative_scores_all[valid_collab_indices]
        )[::-1][:top_k]
    ]

    collaborative_candidates = set(
        movie_ids[collab_top_indices]
    )

    # Get top also-watched candidates
    also_watched_movie_ids = movie_ids_aw[
        ~np.isin(movie_ids_aw, list(rated_set))
    ]

    also_watched_indices = np.array([
        movie_to_index_aw[movie]
        for movie in also_watched_movie_ids
    ])

    also_watched_top_indices = also_watched_indices[
        np.argsort(
            also_watched_scores_all[also_watched_indices]
        )[::-1][:top_k]
    ]

    also_watched_candidates = set(
        movie_ids_aw[also_watched_top_indices]
    )

    # Combine candidates from all three models
    candidate_movie_ids = (
        content_candidates
        | collaborative_candidates
        | also_watched_candidates
    )

    # Remove the selected movie
    candidate_movie_ids.discard(movie_id)

    return np.array(list(candidate_movie_ids))

### Test the candidate pool

Now let's see how many candidates the three models actually produce for User 1 + Matrix (2571).

In [30]:
# Test the new multi-source candidate pool for User 1 and The Matrix

candidate_movie_ids = generate_hybrid_candidates(
    user_id=1,
    movie_id=2571,
    top_k=100
)

print("Number of hybrid candidates:", len(candidate_movie_ids))
print("Selected movie excluded:", 2571 not in candidate_movie_ids)

Number of hybrid candidates: 266
Selected movie excluded: True


### Build the improved hybrid recommender

old hybrid_recommend() intact for now and create a new version called hybrid_recommend_v2().

In [33]:
# Create the improved hybrid recommender with safe handling of movies missing from a model

def hybrid_recommend_v2(user_id, movie_id, n=10, top_k=100):
    if user_id not in user_to_index:
        return "User ID not found."

    if movie_id not in movie_to_index:
        return "Movie ID not found."

    candidate_movie_ids = generate_hybrid_candidates(
        user_id=user_id,
        movie_id=movie_id,
        top_k=top_k
    )

    if isinstance(candidate_movie_ids, str):
        return candidate_movie_ids

    # Calculate content scores
    content_rows = movies_content.index[
        movies_content["movieId"] == movie_id
    ]

    content_index = content_rows[0]

    content_scores_all = cosine_similarity(
        tfidf_matrix[content_index],
        tfidf_matrix
    )[0]

    content_index_map = dict(
        zip(
            movies_content["movieId"],
            movies_content.index
        )
    )

    content_scores = np.array([
        content_scores_all[content_index_map[movie]]
        if movie in content_index_map else 0.0
        for movie in candidate_movie_ids
    ])

    # Calculate collaborative scores safely
    user_index = user_to_index[user_id]

    collaborative_scores_all = (
        user_factors[user_index] @ movie_factors.T
    )

    collaborative_scores = np.array([
        collaborative_scores_all[movie_to_index[movie]]
        if movie in movie_to_index else 0.0
        for movie in candidate_movie_ids
    ])

    # Calculate also-watched scores
    cooccurrence_counts = get_cooccurrence_counts(movie_id)

    selected_movie_users = movie_user_counts.loc[movie_id]

    also_watched_scores_all = (
        cooccurrence_counts
        / np.sqrt(
            selected_movie_users
            * movie_user_counts.reindex(movie_ids_aw).to_numpy()
        )
    )

    also_watched_scores_all[~np.isfinite(also_watched_scores_all)] = 0

    also_watched_index_map = {
        movie_id_aw: index
        for index, movie_id_aw in enumerate(movie_ids_aw)
    }

    also_watched_scores = np.array([
        also_watched_scores_all[
            also_watched_index_map[movie]
        ]
        if movie in also_watched_index_map else 0.0
        for movie in candidate_movie_ids
    ])

    # Normalize each signal
    content_normalized = normalize_scores(content_scores)
    collaborative_normalized = normalize_scores(collaborative_scores)
    also_watched_normalized = normalize_scores(also_watched_scores)

    # Combine the three normalized signals
    hybrid_scores = (
        CONTENT_WEIGHT * content_normalized
        + COLLAB_WEIGHT * collaborative_normalized
        + ALSO_WATCHED_WEIGHT * also_watched_normalized
    )

    # Select the highest hybrid scores
    top_indices = np.argsort(hybrid_scores)[::-1][:n]

    recommended_movie_ids = candidate_movie_ids[top_indices]

    recommendations = movies_content[
        movies_content["movieId"].isin(recommended_movie_ids)
    ][["movieId", "title", "genres_clean"]].copy()

    score_map = dict(
        zip(
            recommended_movie_ids,
            hybrid_scores[top_indices]
        )
    )

    recommendations["hybrid_score"] = (
        recommendations["movieId"].map(score_map)
    )

    recommendations = (
        recommendations
        .set_index("movieId")
        .loc[recommended_movie_ids]
        .reset_index()
    )

    return recommendations

### Compare V1 and V2

In [34]:
# Compare the original hybrid recommender with the new multi-source version

old_recommendations = hybrid_recommend(
    user_id=1,
    movie_id=2571,
    n=10
)

new_recommendations = hybrid_recommend_v2(
    user_id=1,
    movie_id=2571,
    n=10,
    top_k=100
)

print("Original hybrid recommendations:")
display(old_recommendations)

print("\nNew multi-source hybrid recommendations:")
display(new_recommendations)

Original hybrid recommendations:


,movieId,title,genres_clean,hybrid_score
0,924,2001: A Space Odyssey (1968),Adventure|Drama|Sci-Fi,0.792166
1,858,"Godfather, The (1972)",Crime|Drama,0.698755
2,2858,American Beauty (1999),Drama|Romance,0.637763
3,1193,One Flew Over the Cuckoo's Nest (1975),Drama,0.619288
4,1214,Alien (1979),Horror|Sci-Fi,0.581140
5,1704,Good Will Hunting (1997),Drama|Romance,0.580195
6,1198,Raiders of the Lost Ark (Indiana Jones and the...,Action|Adventure,0.576815
7,750,Dr. Strangelove or: How I Learned to Stop Worr...,Comedy|War,0.565532
8,1240,"Terminator, The (1984)",Action|Sci-Fi|Thriller,0.560501
9,318,"Shawshank Redemption, The (1994)",Crime|Drama,0.529131



New multi-source hybrid recommendations:


,movieId,title,genres_clean,hybrid_score
0,924,2001: A Space Odyssey (1968),Adventure|Drama|Sci-Fi,0.682196
1,858,"Godfather, The (1972)",Crime|Drama,0.674178
2,2858,American Beauty (1999),Drama|Romance,0.619046
3,1193,One Flew Over the Cuckoo's Nest (1975),Drama,0.615571
4,1198,Raiders of the Lost Ark (Indiana Jones and the...,Action|Adventure,0.599398
5,1704,Good Will Hunting (1997),Drama|Romance,0.597312
6,750,Dr. Strangelove or: How I Learned to Stop Worr...,Comedy|War,0.583684
7,1214,Alien (1979),Horror|Sci-Fi,0.580430
8,6365,"Matrix Reloaded, The (2003)",Action|Adventure|Sci-Fi|Thriller|IMAX,0.564408
9,318,"Shawshank Redemption, The (1994)",Crime|Drama,0.556975


### Inspect the three signals in version 2

In [36]:
# Inspect how each recommendation signal contributes to the new hybrid ranking

def inspect_hybrid_v2(user_id, movie_id, n=10, top_k=100):
    candidate_movie_ids = generate_hybrid_candidates(
        user_id=user_id,
        movie_id=movie_id,
        top_k=top_k
    )

    if isinstance(candidate_movie_ids, str):
        return candidate_movie_ids

    # Content scores
    content_index = movies_content.index[
        movies_content["movieId"] == movie_id
    ][0]

    content_scores_all = cosine_similarity(
        tfidf_matrix[content_index],
        tfidf_matrix
    )[0]

    content_index_map = dict(
        zip(
            movies_content["movieId"],
            movies_content.index
        )
    )

    content_scores = np.array([
        content_scores_all[content_index_map[movie]]
        if movie in content_index_map else 0.0
        for movie in candidate_movie_ids
    ])

    # Collaborative scores
    user_index = user_to_index[user_id]

    collaborative_scores_all = (
        user_factors[user_index] @ movie_factors.T
    )

    collaborative_scores = np.array([
        collaborative_scores_all[movie_to_index[movie]]
        if movie in movie_to_index else 0.0
        for movie in candidate_movie_ids
    ])

    # Also-watched scores
    cooccurrence_counts = get_cooccurrence_counts(movie_id)

    selected_movie_users = movie_user_counts.loc[movie_id]

    also_watched_scores_all = (
        cooccurrence_counts
        / np.sqrt(
            selected_movie_users
            * movie_user_counts.reindex(movie_ids_aw).to_numpy()
        )
    )

    also_watched_scores_all[~np.isfinite(also_watched_scores_all)] = 0

    also_watched_index_map = {
        movie_id_aw: index
        for index, movie_id_aw in enumerate(movie_ids_aw)
    }

    also_watched_scores = np.array([
        also_watched_scores_all[
            also_watched_index_map[movie]
        ]
        if movie in also_watched_index_map else 0.0
        for movie in candidate_movie_ids
    ])

    # Normalize the three signals
    content_normalized = normalize_scores(content_scores)
    collaborative_normalized = normalize_scores(collaborative_scores)
    also_watched_normalized = normalize_scores(also_watched_scores)

    # Calculate the hybrid score
    hybrid_scores = (
        CONTENT_WEIGHT * content_normalized
        + COLLAB_WEIGHT * collaborative_normalized
        + ALSO_WATCHED_WEIGHT * also_watched_normalized
    )

    # Select the highest hybrid scores
    top_indices = np.argsort(hybrid_scores)[::-1][:n]

    recommended_movie_ids = candidate_movie_ids[top_indices]

    recommendations = movies_content[
        movies_content["movieId"].isin(recommended_movie_ids)
    ][[
        "movieId",
        "title",
        "genres_clean"
    ]].copy()

    score_map = dict(
        zip(
            recommended_movie_ids,
            hybrid_scores[top_indices]
        )
    )

    content_map = dict(
        zip(
            recommended_movie_ids,
            content_normalized[top_indices]
        )
    )

    collaborative_map = dict(
        zip(
            recommended_movie_ids,
            collaborative_normalized[top_indices]
        )
    )

    also_watched_map = dict(
        zip(
            recommended_movie_ids,
            also_watched_normalized[top_indices]
        )
    )

    recommendations["content_score"] = (
        recommendations["movieId"].map(content_map)
    )

    recommendations["collaborative_score"] = (
        recommendations["movieId"].map(collaborative_map)
    )

    recommendations["also_watched_score"] = (
        recommendations["movieId"].map(also_watched_map)
    )

    recommendations["hybrid_score"] = (
        recommendations["movieId"].map(score_map)
    )

    recommendations = (
        recommendations
        .set_index("movieId")
        .loc[recommended_movie_ids]
        .reset_index()
    )

    return recommendations

In [37]:
# Display the individual recommendation signals behind the V2 hybrid ranking

hybrid_v2_details = inspect_hybrid_v2(
    user_id=1,
    movie_id=2571,
    n=10,
    top_k=100
)

display(hybrid_v2_details)

,movieId,title,genres_clean,content_score,collaborative_score,also_watched_score,hybrid_score
0,924,2001: A Space Odyssey (1968),Adventure|Drama|Sci-Fi,0.285407,0.922487,0.676652,0.682196
1,858,"Godfather, The (1972)",Crime|Drama,0.025595,1.000000,0.832496,0.674178
2,2858,American Beauty (1999),Drama|Romance,0.140791,0.799351,0.885666,0.619046
3,1193,One Flew Over the Cuckoo's Nest (1975),Drama,0.046869,0.923974,0.697614,0.615571
4,1198,Raiders of the Lost Ark (Indiana Jones and the...,Action|Adventure,0.036016,0.816436,0.901875,0.599398
5,1704,Good Will Hunting (1997),Drama|Romance,0.040037,0.843880,0.816806,0.597312
6,750,Dr. Strangelove or: How I Learned to Stop Worr...,Comedy|War,0.019763,0.905389,0.625303,0.583684
7,1214,Alien (1979),Horror|Sci-Fi,0.138081,0.760841,0.792925,0.580430
8,6365,"Matrix Reloaded, The (2003)",Action|Adventure|Sci-Fi|Thriller|IMAX,1.000000,0.222636,0.765453,0.564408
9,318,"Shawshank Redemption, The (1994)",Crime|Drama,0.115366,0.678216,0.916285,0.556975


### Save the hybrid configuration

In [38]:
# Save the hybrid model weights and candidate-pool settings for later use

hybrid_config = {
    "content_weight": CONTENT_WEIGHT,
    "collaborative_weight": COLLAB_WEIGHT,
    "also_watched_weight": ALSO_WATCHED_WEIGHT,
    "candidate_top_k": CANDIDATE_TOP_K
}

joblib.dump(
    hybrid_config,
    MODELS_DIR / "hybrid_config.joblib"
)

print("Hybrid configuration saved:")
print(hybrid_config)

Hybrid configuration saved:
{'content_weight': 0.3, 'collaborative_weight': 0.5, 'also_watched_weight': 0.2, 'candidate_top_k': 100}


## Conclusion

The hybrid movie recommendation system was successfully developed by combining three complementary recommendation signals:

* Content-based similarity using movie genres and user-generated tags
* Collaborative filtering based on user-rating behavior
* Also-watched/rated similarity based on movie co-occurrence

The initial hybrid weights were set to 30% content, 50% collaborative filtering, and 20% also-watched/rated similarity. These weights are treated as initial configuration values rather than optimized parameters.

A multi-source candidate-generation strategy was also implemented. Instead of selecting candidates only from collaborative filtering, the system combines candidates from all three recommendation approaches before calculating the final hybrid score.

Testing with different users demonstrated that the hybrid recommender produces personalized recommendations while incorporating both movie-content similarity and behavioral signals.

The hybrid configuration was saved to `models/hybrid_config.joblib` for reuse in the final application.

The next stage is to evaluate the recommendation models quantitatively using a separate evaluation procedure.
